Ячейка 1: Инициализация проекта и импорт библиотек

In [1]:
import os
import json
import re
import pandas as pd
from pathlib import Path

# Определяем пути к данным
RAW_DATA_DIR = Path("../data/raw_things/")
OUTPUT_DIR = Path("../data/output/")

# Создаем папку для выгрузки, если её еще нет
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_file = OUTPUT_DIR / "result.csv"

print("Библиотеки импортированы. Рабочие директории настроены.")

Библиотеки импортированы. Рабочие директории настроены.


Ячейка 2: Загрузка словарей-дешифраторов (Имена и Запчасти)

In [2]:
# Загружаем переводчик имен
name_parts_path = RAW_DATA_DIR / "NexusConfigStoreInventoryNamePart.json"
with open(name_parts_path, "r", encoding="utf-8") as f:
    name_parts_data = json.load(f)

# Загружаем базу запчастей
inv_parts_path = RAW_DATA_DIR / "NexusConfigStoreInventory_Parts.json"
with open(inv_parts_path, "r", encoding="utf-8") as f:
    inv_parts_data = json.load(f)

print(f"Словари загружены! Доступно переводных имен: {len(name_parts_data)}")

Словари загружены! Доступно переводных имен: 614


Ячейка 3: Инициализация базовой таблицы легендарного снаряжения

In [3]:
legendary_items = []

# Проходимся по всем категориям в файле запчастей (например, "2 | DAD_PS", "3 | JAK_PS")
for category, parts_dict in inv_parts_data.items():
    # Пропускаем общую категорию Weapon, нас интересуют конкретные пушки/снаряжение
    if "Weapon" in category:
        continue
        
    for index_id, part_path in parts_dict.items():
        # Если в пути запчасти или компонента есть упоминание легендарности
        if "comp_05_legendary" in part_path:
            # Вытаскиваем системный код предмета (например: DAD_PS.comp_05_legendary_Zipgun)
            item_code = part_path
            
            # Определяем тип предмета по категории (например, DAD_PS -> PS -> Pistol)
            item_type_suffix = category.split("_")[-1] if "_" in category else "Unknown"
            
            # Записываем базовые данные предмета
            legendary_items.append({
                "Item_Code": item_code,
                "Internal_Category": category,
                "Type": item_type_suffix,
                "Rarity": "Legendary",
                "Manufacturer": "Unknown",
                "Display_Name": "Unknown",
                "Drop_Source": "Unknown",
                "Drop_Weight": "-",
                "Elements": "Physical Only",
                "Possible_Parts": ""
            })

# Создаем DataFrame
df = pd.DataFrame(legendary_items)

# Убираем дубликаты, если они случайно попали при сканировании
df = df.drop_duplicates(subset=["Item_Code"]).reset_index(drop=True)

print(f"Инициализация завершена. Найдено легендарных предметов: {len(df)}")
# Посмотрим на первые 5 строк нашей будущей таблицы
df.head()

Инициализация завершена. Найдено легендарных предметов: 333


,Item_Code,Internal_Category,Type,Rarity,Manufacturer,Display_Name,Drop_Source,Drop_Weight,Elements,Possible_Parts
0,DAD_PS.comp_05_legendary_Zipgun,2 | DAD_PS,PS,Legendary,Unknown,Unknown,Unknown,-,Physical Only,
1,DAD_PS.comp_05_legendary,2 | DAD_PS,PS,Legendary,Unknown,Unknown,Unknown,-,Physical Only,
2,DAD_PS.comp_05_legendary_rangefinder,2 | DAD_PS,PS,Legendary,Unknown,Unknown,Unknown,-,Physical Only,
3,DAD_PS.comp_05_legendary_soulsurvivor,2 | DAD_PS,PS,Legendary,Unknown,Unknown,Unknown,-,Physical Only,
4,JAK_PS.comp_05_legendary,3 | JAK_PS,PS,Legendary,Unknown,Unknown,Unknown,-,Physical Only,


Ячейка 4: Определение производителей и красивых типов снаряжения

In [4]:
# Словарь для перевода аббревиатур типов оружия в красивые английские названия
type_mapping = {
    "PS": "Pistol",
    "SR": "Sniper Rifle",
    "AR": "Assault Rifle",
    "SG": "Shotgun",
    "SMG": "Submachine Gun",
    "HW": "Heavy Weapon",
    "SHIELD": "Shield",
    "GRENADE": "Grenade",
    "CLASSMOD": "Class Mod",
    "ARTIFACT": "Artifact"
}

# Словарь соответствия префиксов и полных названий производителей, который мы утвердили
mfr_mapping = {
    "DAD": "Daedalus",
    "ORD": "Order",
    "BORG": "Ripper",
    "BOR": "Ripper",
    "JAK": "Jakobs",
    "VLA": "Vladof",
    "MAL": "Maliwan",
    "HYP": "Hyperion",
    "TED": "Tediore",
    "TOR": "Torgue",
    "COV": "CoV",
    "ATL": "Atlas"
}

def determine_manufacturer(item_code):
    # Извлекаем префикс перед первым нижним подчеркиванием (например, DAD_PS... -> DAD)
    prefix = item_code.split("_")[0].upper()
    # Очищаем от возможных системных кавычек
    prefix = prefix.replace("INV'", "").replace("'", "")
    return mfr_mapping.get(prefix, "Unknown")

# Применяем сопоставление производителей
df["Manufacturer"] = df["Item_Code"].apply(determine_manufacturer)

# Переводим сокращения типов в красивые английские названия
df["Type"] = df["Type"].str.upper().map(type_mapping).fillna(df["Type"])

print("Производители и типы снаряжения успешно определены!")
# Посмотрим, как теперь выглядят эти колонки
df[["Item_Code", "Type", "Manufacturer"]].head()

Производители и типы снаряжения успешно определены!


,Item_Code,Type,Manufacturer
0,DAD_PS.comp_05_legendary_Zipgun,Pistol,Daedalus
1,DAD_PS.comp_05_legendary,Pistol,Daedalus
2,DAD_PS.comp_05_legendary_rangefinder,Pistol,Daedalus
3,DAD_PS.comp_05_legendary_soulsurvivor,Pistol,Daedalus
4,JAK_PS.comp_05_legendary,Pistol,Jakobs


Ячейка 5: Расшифровка оригинальных английских названий предметов

In [5]:
def resolve_display_name(item_code, name_parts):
    # Ищем всё, что идет после "comp_05_legendary_" (или просто "legendary_")
    match = re.search(r"comp_05_legendary_(.*)", item_code, re.IGNORECASE)
    if not match:
        match = re.search(r"legendary_(.*)", item_code, re.IGNORECASE)
        
    if match:
        raw_name = match.group(1).lower().replace("_", "") # Приводим к единому виду (например, "zipgun")
        
        # Ищем строгое соответствие в ключах name_parts (где ключи типа "np_absolution")
        for np_key, np_val in name_parts.items():
            # Очищаем ключ от префикса "np_" и нижних подчеркиваний
            clean_np_key = np_key.lower().replace("np_", "").replace("_", "")
            
            # Если очищенные ключи полностью совпадают
            if clean_np_key == raw_name:
                # Забираем PartName, если его нет — пишем Unknown
                return np_val.get("fields", {}).get("PartName", "Unknown")
                
    return "Unknown"

# Применяем строгую функцию расшифровки имен
df["Display_Name"] = df.apply(lambda row: resolve_display_name(row["Item_Code"], name_parts_data), axis=1)

print("Оригинальные названия предметов успешно расшифрованы!")
# Посмотрим на результат
df[["Item_Code", "Type", "Manufacturer", "Display_Name"]].head(10)

Оригинальные названия предметов успешно расшифрованы!


,Item_Code,Type,Manufacturer,Display_Name
0,DAD_PS.comp_05_legendary_Zipgun,Pistol,Daedalus,Zipper
1,DAD_PS.comp_05_legendary,Pistol,Daedalus,Unknown
2,DAD_PS.comp_05_legendary_rangefinder,Pistol,Daedalus,Rangefinder
3,DAD_PS.comp_05_legendary_soulsurvivor,Pistol,Daedalus,Soul Survivor
4,JAK_PS.comp_05_legendary,Pistol,Jakobs,Unknown
5,JAK_PS.comp_05_legendary_kingsgambit,Pistol,Jakobs,King's Gambit
6,JAK_PS.comp_05_legendary_phantom_flame,Pistol,Jakobs,Phantom Flame
7,JAK_PS.comp_05_legendary_QuickDraw,Pistol,Jakobs,San Saba Songbird
8,JAK_PS.comp_05_legendary_seventh_sense,Pistol,Jakobs,Seventh Sense
9,JAK_PS.comp_05_legendary_shalashaska,Pistol,Jakobs,Shalashaska


Ячейка 6: Привязка боссов-источников и шансов выпадения

In [6]:
# Загружаем базу пулов добычи
item_pool_list_path = RAW_DATA_DIR / "NexusConfigStoreItemPoolList.json"
with open(item_pool_list_path, "r", encoding="utf-8") as f:
    item_pool_list_data = json.load(f)

# Словари для быстрого поиска сопоставлений
drop_sources = {}
drop_weights = {}

# Вспомогательная функция очистки Handle (теперь строго проверяет, что это строка)
def clean_handle(handle):
    if not handle or not isinstance(handle, str):
        return ""
    return handle.lower().replace("inv'", "").replace("'", "").strip()

# Сканируем всю базу ItemPoolList
for list_key, list_val in item_pool_list_data.items():
    # Красиво форматируем название босса/источника (например, ItemPoolList_Arjay -> Arjay)
    boss_name = list_key.replace("ItemPoolList_", "").replace("_", " ").title()
    
    item_pools = list_val.get("fields", {}).get("ItemPools", [])
    for pool_entry in item_pools:
        itempool = pool_entry.get("itempool", {})
        item_data = itempool.get("item", {})
        
        # Вариант 1: Вложенный инстанс пула (bInstance == True)
        if item_data.get("bInstance") and "Instance" in item_data:
            instance = item_data["Instance"] or {}
            items_in_pool = instance.get("items", [])
            
            for pool_item in items_in_pool:
                inner_item = pool_item.get("item", {}).get("item", {})
                handle = inner_item.get("Handle")
                
                if handle:
                    cleaned_h = clean_handle(handle)
                    # Пропускаем пустые и нестроковые значения
                    if cleaned_h:  
                        weight_val = pool_item.get("Weight", {}).get("constant", 1.0)
                        
                        if cleaned_h not in drop_sources:
                            drop_sources[cleaned_h] = []
                            drop_weights[cleaned_h] = []
                        
                        drop_sources[cleaned_h].append(boss_name)
                        drop_weights[cleaned_h].append(str(weight_val))
                        
        # Вариант 2: Прямая ссылка (bInstance == False)
        else:
            handle = item_data.get("Handle")
            if handle:
                cleaned_h = clean_handle(handle)
                # Пропускаем пустые и нестроковые значения
                if cleaned_h:  
                    prob_val = pool_entry.get("probability", {}).get("constant", 1.0)
                    
                    if cleaned_h not in drop_sources:
                        drop_sources[cleaned_h] = []
                        drop_weights[cleaned_h] = []
                    
                    drop_sources[cleaned_h].append(boss_name)
                    drop_weights[cleaned_h].append(str(prob_val))

# Функции сопоставления для DataFrame
def get_drop_source(item_code):
    cleaned_code = clean_handle(item_code)
    if not cleaned_code:
        return "World Drop Only"
    sources = drop_sources.get(cleaned_code, [])
    # Если босс нашелся, пишем его имя, иначе это World Drop (падает отовсюду)
    return ", ".join(set(sources)) if sources else "World Drop Only"

def get_drop_weight(item_code):
    cleaned_code = clean_handle(item_code)
    if not cleaned_code:
        return "-"
    weights = drop_weights.get(cleaned_code, [])
    return ", ".join(weights) if weights else "-"

df["Drop_Source"] = df["Item_Code"].apply(get_drop_source)
df["Drop_Weight"] = df["Item_Code"].apply(get_drop_weight)

# Выведем в качестве примера предметы, у которых нашелся конкретный босс
boss_drops = df[df["Drop_Source"] != "World Drop Only"]
print(f"Источники успешно привязаны! Найдено предметов с уникальным источником: {len(boss_drops)}")
boss_drops[["Display_Name", "Type", "Manufacturer", "Drop_Source", "Drop_Weight"]].head(10)

Источники успешно привязаны! Найдено предметов с уникальным источником: 188


,Display_Name,Type,Manufacturer,Drop_Source,Drop_Weight
0,Zipper,Pistol,Daedalus,"Upgradedelectimole, Upgradedelectimole Trueboss","1.0, 1.0"
2,Rangefinder,Pistol,Daedalus,"Firstcorrupt, Firstcorrupt Trueboss","1.0, 1.0"
3,Soul Survivor,Pistol,Daedalus,Dronecaptain,1.0
5,King's Gambit,Pistol,Jakobs,"Firstcorrupt, Firstcorrupt Trueboss","1.0, 1.0"
6,Phantom Flame,Pistol,Jakobs,"Bango Trueboss, Pango, Pango Trueboss, Bango","1.0, 1.0, 1.0, 1.0"
7,San Saba Songbird,Pistol,Jakobs,"Rockandroll Trueboss, Rockandroll","1.0, 1.0"
8,Seventh Sense,Pistol,Jakobs,"Sidecity Psycho Trueboss, Sidecity Psycho","1.0, 1.0"
9,Shalashaska,Pistol,Jakobs,Ordonite Pgg Activity,1.0
10,Shoals,Pistol,Jakobs,Tuba Terra,1.0
11,Lucky Clover,Pistol,Order,"Kotolieutenant, Kotolieutenant Trueboss","1.0, 1.0"


Ячейка 7: Сбор возможных модулей для каждого предмета

In [7]:
def get_possible_parts(category, parts_dict):
    if category not in parts_dict:
        return "-"
    
    cat_parts = parts_dict[category]
    parts_list = []
    
    for index_id, part_path in cat_parts.items():
        # Фильтруем только физические запчасти, игнорируя маркеры сборки пушек (.comp_)
        if ".part_" in part_path or "_part" in part_path:
            # Очищаем от префиксов (например, "JAK_PS.part_barrel_01" -> "barrel_01")
            part_name = part_path.split(".")[-1].replace("part_", "")
            parts_list.append(part_name)
            
    return ", ".join(sorted(list(set(parts_list)))) if parts_list else "-"

df["Possible_Parts"] = df.apply(lambda row: get_possible_parts(row["Internal_Category"], inv_parts_data), axis=1)

print("Возможные модули для оружия успешно собраны!")
df[["Display_Name", "Manufacturer", "Possible_Parts"]].head(5)

Возможные модули для оружия успешно собраны!


,Display_Name,Manufacturer,Possible_Parts
0,Zipper,Daedalus,"barrel_01, barrel_01_a, barrel_01_b, barrel_01..."
1,Unknown,Daedalus,"barrel_01, barrel_01_a, barrel_01_b, barrel_01..."
2,Rangefinder,Daedalus,"barrel_01, barrel_01_a, barrel_01_b, barrel_01..."
3,Soul Survivor,Daedalus,"barrel_01, barrel_01_a, barrel_01_b, barrel_01..."
4,Unknown,Jakobs,"barrel_01, barrel_01_a, barrel_01_b, barrel_01..."


Ячейка 8: Экспорт итоговой таблицы в CSV

In [8]:
# Создаем копию, чтобы не повредить исходный DataFrame
final_df = df.copy()

# Переименовываем колонки для красивой выгрузки
final_df = final_df.rename(columns={
    "Display_Name": "Name",
    "Drop_Source": "Drop Source",
    "Drop_Weight": "Drop Weight",
    "Possible_Parts": "Possible Parts",
    "Item_Code": "Item Code"
})

# Задаем порядок колонок
columns_order = [
    "Name", "Rarity", "Type", "Manufacturer", 
    "Drop Source", "Drop Weight", "Possible Parts", "Item Code"
]
final_df = final_df[columns_order]

# Сохраняем в CSV-файл
final_df.to_csv(output_file, index=False, encoding="utf-8")

print(f"Таблица успешно экспортирована в: {output_file}")
print(f"Итоговый размер таблицы: {final_df.shape[0]} строк на {final_df.shape[1]} колонок.")

Таблица успешно экспортирована в: ../data/output/result.csv
Итоговый размер таблицы: 333 строк на 8 колонок.
